In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import SMOTENC
from sklearn.decomposition import PCA
from imblearn.pipeline import Pipeline
from sklearn.metrics import recall_score
from sklearn.metrics import make_scorer
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer, MinMaxScaler
from sklearn.base import BaseEstimator, TransformerMixin

In [28]:
df = pd.read_csv('/Users/Tudor.Orosz/Documents/School/Thesis/Extract Data Athena/Dataframe marketing and platform.csv')

df = df.drop(["id", "accountid", "account_sales_unit__c", "sales_unit_group__c", "name", "createddate"], axis = 1)

In [29]:
X = df.drop("opportunityid", axis = 1)
y = df.loc[:, ["opportunityid"]]

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=548)

X_train2, X_val, y_train2, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=548)

In [31]:
# Define the columns to be one-hot encoded
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Specify numerical variables
X_numb = X_train.select_dtypes(include=['int64', 'float64'])

# Define PCA indices
indices_pca = [i for i, col in enumerate(X.columns) if col in X_numb.columns]

In [32]:
# Define the pipeline
numeric_transformer = Pipeline(steps=[
    ('scaler', MinMaxScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, X_train2.select_dtypes(include=['int64', 'float64']).columns.tolist()),
        ('cat', categorical_transformer, categorical_cols),
    ])

# Add the upsampler and the pipeline to a machine learning model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # ('over', SMOTENC(categorical_features=[1, 2], random_state=548)),
    ('regressor', LogisticRegression(random_state=548))
])

parameters = {
    'regressor__C': [0.1, 1.0, 10.0],
    'regressor__penalty': ['l1', 'l2', 'elasticnet', 'none'],
    'regressor__solver': ['lbfgs', 'liblinear', 'saga']
}

In [33]:
# Set up the grid search object
grid_search = GridSearchCV(model, parameters, cv=10)

In [34]:
# Fit the grid search object to the training data
grid_search.fit(X_train2, y_train2.values.ravel())

/Users/Tudor.Orosz/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1173: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
/Users/Tudor.Orosz/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/Users/Tudor.Orosz/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/Tudor.Orosz/miniconda

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('scaler',
                                                                                          MinMaxScaler())]),
                                                                         ['M_WBN_Attended',
                                                                          'M_WBN_Attended '
                                                                          'On-Demand',
                                                                          'M_Other_Filled '
                                                                          'Out '
                                                                          'Form',
                                                                          'M_TS_Attended',
                                                                          'M_TS_Visited '
                                                                          'Booth',
                                                                          'M_LE_Attended',
                                                                          'M_WBN_Attended '
                                                                          'On-demand',
                                                                          'M_VLE_Attended',
                                                                          'M_TS_Visited '
                                                                          'booth',
                                                                          'M_SE_Atten...
                                                                          'Asset', ...]),
                                                                        ('cat',
                                                                         Pipeline(steps=[('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['gtm_segment__c',
                                                                          'gtm_industry__c',
                                                                          'region__c',
                                                                          'country_grouping__c'])])),
                                       ('regressor',
                                        LogisticRegression(random_state=548))]),
             param_grid={'regressor__C': [0.1, 1.0, 10.0],
                         'regressor__penalty': ['l1', 'l2', 'elasticnet',
                                                'none'],
                         'regressor__solver': ['lbfgs', 'liblinear', 'saga']})

In [35]:
# Get the best parameters from the grid search
best_params = grid_search.best_params_
print(best_params)

{'regressor__C': 0.1, 'regressor__penalty': 'l2', 'regressor__solver': 'lbfgs'}


In [36]:
y_pred = grid_search.predict(X_val)
y_pred

array([0, 0, 0, ..., 0, 0, 0])

In [37]:
grid_search.best_estimator_.feature_importances_

AttributeError: 'Pipeline' object has no attribute 'feature_importances_'

In [ ]:
# # Create a dictionary of feature names and importance scores
# feature_importance = dict(zip(feature_names, importance_scores))

# # Sort the features by importance score (descending order)
# sorted_features = sorted(feature_importance.items(), key=lambda x: abs(x[1]), reverse=True)

# # Print the top 10 features and their importance scores
# print("Top 10 Features:")
# for feature, importance in sorted_features[:10]:
#     print(f"{feature}: {importance}")

In [ ]:
y_proba = grid_search.predict_proba(X_val)
y_pred_new = (y_proba[:, 1] > 0.2).astype(int)

In [ ]:
# Compute the confusion matrix
cm = confusion_matrix(y_val, y_pred_new)

# Print the confusion matrix with labels
print("Confusion matrix:")
print("           Predicted")
print("           0   1")
print("Actual 0  ", cm[0][0], " ", cm[0][1])
print("       1  ", cm[1][0], " ", cm[1][1])

Confusion matrix:
           Predicted
           0   1
Actual 0   10084   67
       1   788   14


In [ ]:
# Calculate the confusion matrix
cm = confusion_matrix(y_val, y_pred_new)

# Extract the true negatives, false positives, false negatives, and true positives
tn, fp, fn, tp = cm.ravel()

# Calculate accuracy, specificity, and sensitivity
accuracy = (tp + tn) / (tp + tn + fp + fn)
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)

# Print the results
print(f"Accuracy: {accuracy:.2f}")
print(f"Specificity: {specificity:.2f}")
print(f"Sensitivity: {sensitivity:.2f}")

Accuracy: 0.92
Specificity: 0.99
Sensitivity: 0.02
